# RG3 정상영역 기반 이상 탐지
가스샘플 모델적합제외. 3이상탐지모델 고정앙상블.

In [1]:
from pathlib import Path
import json
import numpy as np,pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.covariance import LedoitWolf
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import average_precision_score,confusion_matrix
root=Path.cwd()
if not (root/'data').exists():root=root.parent
nb=json.loads((root/'notebooks/22_rg3_nested_signal_ensemble.ipynb').read_text(encoding='utf-8'))
exec(''.join(nb['cells'][1]['source']).split('scores=[];thresholds=[];foldlogs=[]')[0])
rawcols=['Max_Injection_Pressure','Average_Back_Pressure','Cushion_Position','Plasticizing_Position','Barrel_Temperature_2','Barrel_Temperature_5','Hopper_Temperature','Mold_Temperature_3']
# Current values and dimensionless local deviations; no new labels or RPM assumptions.
views={'state8':d[rawcols].to_numpy(float)}
Z=[]
for c in rawcols:
    floor=.1 if 'Temperature' in c else .01 if 'Position' in c else 1.
    Z.append(d[c+'__dev10'].to_numpy(float)/np.maximum(d[c+'__past10_std'].to_numpy(float),floor))
views['local8']=np.column_stack(Z)
predcache={};fitlog=[]
def anomaly_scores(view,ref,q):
    key=(view,tuple(ref),tuple(q))
    if key in predcache:return predcache[key]
    assert set(groups.iloc[ref]).isdisjoint(groups.iloc[q])
    normal=ref[y[ref]==0]
    if len(normal)<5:return np.zeros((len(q),3))
    imputer=SimpleImputer(strategy='median',keep_empty_features=True)
    a=imputer.fit_transform(views[view][normal]);b=imputer.transform(views[view][q])
    scaler=RobustScaler().fit(a);a=scaler.transform(a);b=scaler.transform(b)
    covariance=LedoitWolf().fit(a)
    forest=IsolationForest(n_estimators=200,max_samples=min(256,len(a)),random_state=42,n_jobs=-1).fit(a)
    nearest=NearestNeighbors(n_neighbors=min(10,len(a))).fit(a)
    scores=np.column_stack([covariance.mahalanobis(b),-forest.score_samples(b),nearest.kneighbors(b)[0].mean(axis=1)])
    fitlog.append(dict(view=view,fit_normal=len(normal),fit_gas=0,query=len(q),fit_blocks='|'.join(groups.iloc[normal].unique()),query_blocks='|'.join(groups.iloc[q].unique())))
    predcache[key]=scores
    return scores
def normal_cdf(ref,q):
    ref=np.sort(ref[np.isfinite(ref)])
    return np.searchsorted(ref,q,side='left')/len(ref) if len(ref) else np.zeros(len(q))
records=[]
for scheme in ['leave_one_block_out','past_blocks_only']:
    for j,block in enumerate(blocks):
        q=np.flatnonzero(groups.eq(block));ref=np.flatnonzero(~groups.eq(block)) if scheme=='leave_one_block_out' else np.flatnonzero(groups.isin(blocks[:j]))
        # Match exactly the baseline evaluation population.
        if len(np.unique(y[ref]))<2:continue
        for view in views:
            out=anomaly_scores(view,ref,q);inside=np.full((len(ref),3),np.nan)
            for ib in groups.iloc[ref].unique():
                mask=groups.iloc[ref].eq(ib).to_numpy()
                if (~mask).any():inside[mask]=anomaly_scores(view,ref[~mask],ref[mask])
            normal=inside[y[ref]==0]
            ranked_out=np.column_stack([normal_cdf(normal[:,k],out[:,k]) for k in range(3)])
            ranked_in=np.column_stack([normal_cdf(normal[:,k],inside[:,k]) for k in range(3)])
            candidates={'three_mean':(ranked_out.mean(axis=1),ranked_in.mean(axis=1)),
'three_consensus':(np.median(ranked_out,axis=1),np.median(ranked_in,axis=1))}
            for k,name in enumerate(['mahalanobis','isolation','nearest']):candidates[name]=(out[:,k],inside[:,k])
            for name,(score,cal) in candidates.items():
                norm=cal[(y[ref]==0)&np.isfinite(cal)]
                for rate in [.005,.01,.02]:
                    cut=float(np.quantile(norm,1-rate,method='higher')) if len(norm) else np.inf
                    for i,z in enumerate(q):records.append(dict(scheme=scheme,view=view,model=name,budget=rate,TimeStamp=d.TimeStamp.iloc[z],block=block,actual=int(y[z]),score=float(score[i]),prediction=int(score[i]>cut),threshold=cut))
        print('Completed normal-only',scheme,block,flush=True)
pp=pd.DataFrame(records);rows=[]
for (scheme,view,name,budget),g in pp.groupby(['scheme','view','model','budget'],sort=False):
    tn,fp,fn,tp=confusion_matrix(g.actual,g.prediction,labels=[0,1]).ravel()
    rows.append(dict(scheme=scheme,view=view,model=name,budget=budget,shots=len(g),gas=int(g.actual.sum()),AP=average_precision_score(g.actual,g.score),TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),F1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0))
metrics=pd.DataFrame(rows)
metrics.to_csv(tables/'rg3_normal_only_metrics.csv',index=False,encoding='utf-8-sig')
pp.to_csv(tables/'rg3_normal_only_predictions.csv',index=False,encoding='utf-8-sig')
pd.DataFrame(fitlog).to_csv(tables/'rg3_normal_only_fit_audit.csv',index=False,encoding='utf-8-sig')
summary=metrics.loc[metrics.budget.eq(.01)].sort_values(['scheme','F1','AP'],ascending=[True,False,False])
lines=['# RG3 정상만 학습하는 이상 탐지: 다른 접근','',
'지도분류RF의가중치/임계값튜닝과별개다. 각바깥학습구간에서라벨정상만선택하고가스는모델적합에서제외한다. 정상선별에는라벨을쓰므로정상샘플학습(one-class/novelty)방식이며비라벨전체정상가정이아니다.',
'8개현재센서(state8)와현재값-이전10샷중앙값을이전10샷표준편차로정규화한8개(local8)를분리해시험한다. 피처는압력/배압/쿠션/계량위치/배럴2/배럴5/호퍼/금형3이다. 결측대체및RobustScaler는모델학습정상만적합한다.',
'LedoitWolfMahalanobis/IsolationForest/10정상최근접평균거리로정상영역이탈을측정한다. 학습내부구간제외정상점수의CDF로척도를맞춰3개평균및중앙값(2개이상동의)앙상블을고정한다. 현재평가정답으로가중치를고르지않는다.',
'기준은학습내부정상상위0.5/1/2%로정하고평가오탐률보장없음. 내부CDF기준표본재사용이며독립확률보정아님. 가스모델학습은안하지만평가모집단을기존과맞추려고학습집합에두클래스가있는구간만평가한다. 구간제외583/가스17, 과거440/가스13.',
'같은자료반복탐색으로독립최종성능이아니다. 실제Run ID가아닌관측구간. 원본이력확보조건. 정상영역이탈을가스원인으로단정하지않는다.',
'','| 평가 | 피처 | 모델/앙상블 | AP | TP | FP | FN | TN | F1 |','|---|---|---|---:|---:|---:|---:|---:|---:|']
for r in summary.itertuples():lines.append(f'| {r.scheme} | {r.view} | {r.model} | {r.AP:.4f} | {r.TP} | {r.FP} | {r.FN} | {r.TN} | {r.F1:.4f} |')
(root/'reports/rg3_normal_only_comparison.md').write_text('\n'.join(lines),encoding='utf-8')
print(summary.to_string(index=False))


             scheme   view           model  budget  shots  gas       AP  TP  FP  FN  TN       F1
leave_one_block_out state8      three_mean    0.01    583   17 0.057956   0   0  17 566 0.000000
leave_one_block_out state8         nearest    0.01    583   17 0.044263   0   0  17 566 0.000000
leave_one_block_out state8     mahalanobis    0.01    583   17 0.041311   0   4  17 562 0.000000
leave_one_block_out state8 three_consensus    0.01    583   17 0.040689   0   0  17 566 0.000000
leave_one_block_out state8       isolation    0.01    583   17 0.037651   0   3  17 563 0.000000
leave_one_block_out local8     mahalanobis    0.01    583   17 0.025295   0   7  17 559 0.000000
leave_one_block_out local8      three_mean    0.01    583   17 0.024901   0   4  17 562 0.000000
leave_one_block_out local8         nearest    0.01    583   17 0.024586   0   3  17 563 0.000000
leave_one_block_out local8       isolation    0.01    583   17 0.024262   0   6  17 560 0.000000
leave_one_block_out local8 thr